# Unit 05: Error Handling, Logging, Testing, JSON, and CSV

This notebook is a guided, runnable version of `student_records_app.py`. It slows the script down into small notebook-friendly examples so you can see **what each idea does**, **why it matters**, and **how to use it in your own project**.

The theme is a tiny study-record tracker:

- A record stores a student name, a topic, minutes studied, and whether the topic is complete.
- Validation catches bad data before it is saved.
- Error handling keeps expected problems from crashing the program.
- Logging records what happened while the program ran.
- JSON and CSV examples show two common file formats.
- Tests prove the helper functions still behave correctly.


## How to Use This Notebook

Run the cells from top to bottom. Most examples are independent, but the later sections reuse the `StudyRecord` class and helper functions defined near the beginning.

Recommended workflow:

1. **Read the markdown first** so you know what the code is trying to demonstrate.
2. **Run the code cell** and inspect the output.
3. **Change one small value** and run the cell again.
4. When an error appears, read the message before fixing it. The message is part of the lesson.

> Notebook tip: if the state gets confusing, choose **Kernel → Restart Kernel and Run All Cells**.


## Learning Goals

By the end of this notebook, you should be able to:

- Create a small data model with `@dataclass`.
- Validate data and raise `ValueError` for invalid input.
- Use `try`, `except`, `else`, and `finally` to handle expected problems.
- Configure `logging` and read log messages.
- Save and load records with JSON.
- Save and load records with CSV.
- Write simple tests with `unittest`.
- Explain when to use JSON, when to use CSV, and when to add logging.


## 1. Create a Data Model

A **data model** describes the shape of the information our program works with. Here, one `StudyRecord` has four fields:

| Field | Type | Example | Meaning |
| --- | --- | --- | --- |
| `student` | `str` | `"Maya"` | Student name |
| `topic` | `str` | `"Logging"` | Study topic |
| `minutes` | `int` | `20` | Minutes spent studying |
| `completed` | `bool` | `True` | Whether the topic was completed |

`@dataclass` automatically creates useful methods such as `__init__` and `__repr__`, which makes small record-style classes easier to write.


In [ ]:
from dataclasses import asdict, dataclass


@dataclass
class StudyRecord:
    """A single study session record."""

    student: str
    topic: str
    minutes: int
    completed: bool


first_record = StudyRecord(student="Maya", topic="Error handling", minutes=30, completed=True)
first_record

### Convert a Dataclass to a Dictionary

JSON and CSV tools usually work with built-in Python data structures such as dictionaries and lists. The `asdict()` helper converts a dataclass object into a dictionary.


In [ ]:
record_dictionary = asdict(first_record)
record_dictionary

## 2. Validate Data Before Saving

Validation means checking that data follows the rules of your program. Our rules are:

- `student` cannot be blank.
- `topic` cannot be blank.
- `minutes` must be greater than `0`.

When data breaks a rule, the function raises `ValueError`. Raising an error is useful because it stops bad data from quietly entering the rest of the program.


In [ ]:
def validate_record(record: StudyRecord) -> None:
    """Raise ValueError when a study record is not valid."""
    if not record.student.strip():
        raise ValueError("student name cannot be blank")
    if not record.topic.strip():
        raise ValueError("topic cannot be blank")
    if record.minutes <= 0:
        raise ValueError("minutes must be greater than 0")


valid_record = StudyRecord("Maya", "Testing", 25, True)
validate_record(valid_record)
print("Valid record accepted!")

### Example: Catch a Validation Error

The next cell intentionally creates an invalid record. Instead of letting the notebook crash, the `try`/`except` block catches the expected `ValueError` and prints a friendly message.


In [ ]:
invalid_record = StudyRecord("Maya", "Testing", 0, True)

try:
    validate_record(invalid_record)
except ValueError as error:
    print(f"Handled expected problem: {error}")

### Add Records Safely

A common pattern is to validate first, then update the data. The `add_record()` function returns a **new list** instead of modifying the original list directly.


In [ ]:
def add_record(records: list[StudyRecord], record: StudyRecord) -> list[StudyRecord]:
    """Validate one record and return a new list containing it."""
    validate_record(record)
    return [*records, record]


records: list[StudyRecord] = []
records = add_record(records, StudyRecord("Maya", "Logging", 20, True))
records = add_record(records, StudyRecord("Maya", "JSON and CSV", 35, False))
records

## 3. Understand `try`, `except`, `else`, and `finally`

Error handling lets your program respond to problems you can predict.

| Part | Runs when... | Common use |
| --- | --- | --- |
| `try` | Always attempted first | Put risky code here |
| `except` | A matching error happens | Recover or show a friendly message |
| `else` | No error happens | Continue with success-only work |
| `finally` | Always runs | Cleanup, closing resources, final messages |

The next function converts text to an integer and demonstrates all four parts.


In [ ]:
def explain_number(text: str) -> None:
    try:
        number = int(text)
    except ValueError:
        print(f"'{text}' is not a valid whole number.")
    else:
        print(f"Success! {number} doubled is {number * 2}.")
    finally:
        print("Finished checking input.\n")


explain_number("12")
explain_number("twelve")

## 4. Logging in a Notebook

`print()` is helpful while learning, but logging is better for real programs because it can save a timeline of events to a file.

Logging levels help describe importance:

| Level | Use it for |
| --- | --- |
| `DEBUG` | Detailed information for troubleshooting |
| `INFO` | Normal program milestones |
| `WARNING` | Something unexpected happened, but the program can continue |
| `ERROR` | A serious operation failed |

In notebooks, logging configuration can be reused from previous runs, so this example uses `force=True` to reset the configuration for this lesson.


In [ ]:
import logging
from pathlib import Path
from tempfile import TemporaryDirectory


def configure_notebook_logging(log_path: Path) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(
        filename=log_path,
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        force=True,
    )


with TemporaryDirectory() as folder:
    log_path = Path(folder) / "logs" / "unit05_notebook.log"
    configure_notebook_logging(log_path)

    logging.info("Notebook logging example started")
    logging.warning("This is a practice warning")
    logging.error("This is a practice error message")

    print(log_path.read_text(encoding="utf-8"))

### Logging While Adding Records

Now we can combine validation, error handling, and logging. Valid records are added. Invalid records are skipped and logged as errors.


In [ ]:
sample_records = [
    StudyRecord("Maya", "Error handling", 30, True),
    StudyRecord("", "Blank student example", 15, False),
    StudyRecord("Maya", "Logging", 20, True),
]

with TemporaryDirectory() as folder:
    log_path = Path(folder) / "unit05_add_records.log"
    configure_notebook_logging(log_path)

    clean_records: list[StudyRecord] = []
    for record in sample_records:
        try:
            clean_records = add_record(clean_records, record)
            logging.info("Added record for %s on %s", record.student, record.topic)
        except ValueError as error:
            logging.error("Could not add record: %s", error)
            print(f"Skipping invalid record: {error}")

    print("Clean records:", clean_records)
    print("\nLog contents:")
    print(log_path.read_text(encoding="utf-8"))

## 5. JSON Round Trip

A **round trip** means saving data and loading it back again.

JSON is a good choice when:

- You want to preserve nested data such as lists and dictionaries.
- You want a file format that is readable by Python, JavaScript, and many other languages.
- Your data is more object-like than spreadsheet-like.

The JSON workflow is:

1. Convert `StudyRecord` objects to dictionaries.
2. Save the list of dictionaries with `json.dumps()` or `json.dump()`.
3. Read the file back.
4. Convert dictionaries back into `StudyRecord` objects.


In [ ]:
import json


def records_to_dicts(records: list[StudyRecord]) -> list[dict[str, object]]:
    return [asdict(record) for record in records]


def parse_bool(value: object) -> bool:
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def dict_to_record(row: dict[str, object]) -> StudyRecord:
    return StudyRecord(
        student=str(row["student"]),
        topic=str(row["topic"]),
        minutes=int(row["minutes"]),
        completed=parse_bool(row["completed"]),
    )


records_as_dicts = records_to_dicts(records)
json_text = json.dumps(records_as_dicts, indent=2)
print(json_text)

In [ ]:
def save_records_json(records: list[StudyRecord], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(records_to_dicts(records), indent=2), encoding="utf-8")


def load_records_json(path: Path) -> list[StudyRecord]:
    try:
        raw_records = json.loads(path.read_text(encoding="utf-8"))
    except FileNotFoundError:
        logging.warning("JSON file was not found: %s", path)
        return []
    except json.JSONDecodeError:
        logging.error("JSON file could not be decoded: %s", path)
        return []

    return [dict_to_record(row) for row in raw_records]


with TemporaryDirectory() as folder:
    json_path = Path(folder) / "study_records.json"
    save_records_json(records, json_path)
    loaded_records = load_records_json(json_path)

    print("Saved JSON file:")
    print(json_path.read_text(encoding="utf-8"))
    print("Loaded records:", loaded_records)
    print("Round trip worked:", loaded_records == records)

### Example: Handle a Missing or Broken JSON File

File input is risky because the file might be missing or damaged. The loader above returns an empty list for two expected problems:

- `FileNotFoundError`: the file does not exist.
- `json.JSONDecodeError`: the file exists but is not valid JSON.


In [ ]:
with TemporaryDirectory() as folder:
    missing_path = Path(folder) / "missing.json"
    broken_path = Path(folder) / "broken.json"
    broken_path.write_text("this is not json", encoding="utf-8")

    print("Missing file result:", load_records_json(missing_path))
    print("Broken file result:", load_records_json(broken_path))

## 6. CSV Preview and Round Trip

CSV stands for **comma-separated values**. It is a good choice when:

- Your data is naturally rows and columns.
- You want to open the file in a spreadsheet tool.
- You want a simple export format.

CSV stores text, so values like `minutes` and `completed` need to be converted back to `int` and `bool` when loading.


In [ ]:
import csv
import io

output = io.StringIO()
writer = csv.DictWriter(output, fieldnames=["student", "topic", "minutes", "completed"])
writer.writeheader()
writer.writerows(records_to_dicts(records))

csv_preview = output.getvalue()
print(csv_preview)

In [ ]:
def save_records_csv(records: list[StudyRecord], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["student", "topic", "minutes", "completed"])
        writer.writeheader()
        writer.writerows(records_to_dicts(records))


def load_records_csv(path: Path) -> list[StudyRecord]:
    try:
        with path.open("r", newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            return [dict_to_record(row) for row in reader]
    except FileNotFoundError:
        logging.warning("CSV file was not found: %s", path)
        return []


with TemporaryDirectory() as folder:
    csv_path = Path(folder) / "study_records.csv"
    save_records_csv(records, csv_path)
    loaded_records = load_records_csv(csv_path)

    print("Saved CSV file:")
    print(csv_path.read_text(encoding="utf-8"))
    print("Loaded records:", loaded_records)
    print("Round trip worked:", loaded_records == records)

## 7. Build Summary Functions

Small helper functions are easier to test than one large program. These functions answer simple questions about the records:

- How many total minutes were studied?
- Which topics were completed?
- What summary should be printed for the user?


In [ ]:
def total_minutes(records: list[StudyRecord]) -> int:
    return sum(record.minutes for record in records)


def completed_topics(records: list[StudyRecord]) -> list[str]:
    return [record.topic for record in records if record.completed]


def average_minutes(records: list[StudyRecord]) -> float:
    if not records:
        return 0.0
    return total_minutes(records) / len(records)


def build_summary(records: list[StudyRecord]) -> str:
    topics = completed_topics(records)
    completed_text = ", ".join(topics) if topics else "None yet"
    return (
        f"Records: {len(records)}\n"
        f"Total minutes: {total_minutes(records)}\n"
        f"Average minutes: {average_minutes(records):.1f}\n"
        f"Completed topics: {completed_text}"
    )


print(build_summary(records))

## 8. Testing Idea

A test checks that a small piece of code behaves the way we expect. Python includes a built-in testing library named `unittest`.

Good beginner tests often check:

- A valid record is accepted.
- An invalid record raises `ValueError`.
- Summary functions return expected values.
- JSON and CSV round trips load the same records that were saved.

The project test file is `tests/test_student_records_app.py`. The next cell shows a small notebook-friendly test suite.


In [ ]:
import unittest


class TestNotebookStudyRecords(unittest.TestCase):
    def test_valid_record_is_added(self) -> None:
        record = StudyRecord("Maya", "Testing", 25, True)
        self.assertEqual(add_record([], record), [record])

    def test_zero_minutes_is_rejected(self) -> None:
        record = StudyRecord("Maya", "Testing", 0, True)
        with self.assertRaises(ValueError):
            add_record([], record)

    def test_summary_helpers(self) -> None:
        test_records = [
            StudyRecord("Maya", "Logging", 20, True),
            StudyRecord("Maya", "CSV", 35, False),
        ]
        self.assertEqual(total_minutes(test_records), 55)
        self.assertEqual(completed_topics(test_records), ["Logging"])
        self.assertEqual(average_minutes(test_records), 27.5)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestNotebookStudyRecords)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## 9. How This Notebook Connects to the Script

The full script uses the same ideas in one terminal program:

```bash
python python_for_intermediate_class/unit05_error_logging_testing_data/student_records_app.py
```

The test file can be run with:

```bash
python -m unittest python_for_intermediate_class.unit05_error_logging_testing_data.tests.test_student_records_app
```

Use the notebook to experiment. Use the `.py` file when you want a repeatable program that writes files and logs in the project folder.


## Practice Challenges

Try these in new cells below this notebook:

1. **Add a new valid record** for a different student.
2. **Create a blank topic** and catch the error message.
3. **Change `average_minutes()`** so it rounds to two decimal places.
4. **Add a new summary helper** named `incomplete_topics()`.
5. **Write a test** for your new helper.
6. **Save your custom records** to JSON and CSV using a temporary directory.

Stretch goal: add a `difficulty` field to `StudyRecord`. After adding the field, update validation, JSON, CSV, and tests.


In [ ]:
# Practice space: write your own experiments here.

